# Walkthrough — Word Embeddings

This notebook calls the service's *real* modules in sequence rather than
reimplementing any of them, so it goes stale loudly rather than quietly.

The pipeline is one stage, not four: `text8` is fetched from the Hub on first use,
so there is no corpus stage to run first. `word_embeddings/etl.py` still exists, but
it builds the Hacker News corpus the sibling content-scoring service consumes and
nothing on this training path calls it.

1. **Corpus and vocabulary** — `corpus.py`, fetched and counted.
2. **Batching** — `data.py`, subsampling and the window batcher.
3. **Training** — `train.py`, a real run over a small slice.
4. **Evaluation** — `benchmarks.py`, the three intrinsic benchmarks.
5. **Serving** — `serving/api.py`, the nearest-neighbour and analogy endpoints.

Run it from this directory; the first cell moves to the service root, which is
where every `make` target runs from and what the config's relative paths assume.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "word-embeddings" and os.path.isdir("notebooks"):
    pass
else:
    os.chdir("../../services/word-embeddings")

print(os.getcwd())

## 1. Corpus and vocabulary

`load_corpus` fetches text8 through the Hub helper, counts it, drops everything
below `min_count`, and encodes it against the surviving vocabulary.

Two decisions worth seeing in the output. Ids are assigned in **descending frequency
order**, so "the N most frequent words" is the slice `[:N]` — which is what every
plot and probe in this service wants. And words below the threshold are **dropped**
rather than mapped to `<UNK>`: one row standing in for every rare word becomes a
high-frequency token that appears beside everything and trains into the middle of
the space.

At `min_count=5` this gives 71,290 words, which is the standard text8 figure — a
useful check that the tokenization matches everyone else's.

In [ ]:
from config import Config
from word_embeddings.corpus import load_corpus

config = Config()
corpus = load_corpus(config)

index_to_word = corpus.index_to_word
print(f"vocabulary: {corpus.vocab_size:,} words   corpus: {len(corpus):,} tokens")
print("most frequent:", [index_to_word[i] for i in range(12)])
print(
    "least frequent:", [index_to_word[i] for i in range(corpus.vocab_size - 5, corpus.vocab_size)]
)

## 2. Subsampling and batching

`keep_probabilities` is Mikolov's discard rule, `P(keep) = (sqrt(z/t) + 1) * (t/z)`.
It is what stops most of the gradient going into `the`, and on this corpus it is the
single highest-leverage setting — the sweep found `t=1e-3` scoring 0.290 against
0.486 at `t=1e-5`.

The batcher keeps the corpus as one tensor and cuts batches out of it with
`Tensor.unfold`, which is a view. The version this replaced materialised every
training example as a Python tuple up front — about 17M of them, before the first
optimizer step.

In [ ]:
from word_embeddings.data import Word2VecBatcher, keep_probabilities

keep = keep_probabilities(corpus.counts, config.subsample_threshold)
for word in ("the", "of", "king", "tributaries"):
    i = corpus.word_to_index[word]
    share = corpus.counts[i].item() / corpus.counts.sum().item()
    print(f"  {word:14} {share:8.5%} of the corpus -> kept {keep[i]:6.2%} of the time")

In [ ]:
import torch

batcher = Word2VecBatcher(
    corpus, config, device=torch.device("cpu"), generator=torch.Generator().manual_seed(0)
)
centre, context, negatives = next(batcher.epoch())

print(
    f"{len(batcher):,} batches/epoch   centre {tuple(centre.shape)}"
    f"   context {tuple(context.shape)}   negatives {tuple(negatives.shape)}"
)

w = config.window_size
print("\none window, centre in brackets:")
print(
    " ",
    " ".join(index_to_word[int(i)] for i in context[0][:w]),
    f"[[{index_to_word[int(centre[0])]}]]",
    " ".join(index_to_word[int(i)] for i in context[0][w:]),
)

# Drawn from the unigram^0.75 distribution, so these are mid-frequency words --
# not all `the`, and not all hapaxes the model already scores near zero.
print("negatives:", [index_to_word[int(i)] for i in negatives[0][:8]])

## 3. Training

`train()` is the real loop. Below it runs over a small slice of the corpus so the
notebook finishes in seconds; a full run is ~75s/epoch on an M1 Max.

Note what it prints first: the **untrained baseline**. Every eval curve carries its
own reference that way, and it is the number that exposed the state this service was
in before — a shipped checkpoint reporting a loss of 10.02, which reads like a
trained model until you know an untrained one reports 11.09.

In [ ]:
from word_embeddings.corpus import Corpus
from word_embeddings.train import random_baseline, train

demo = Config(
    architecture="skipgram",
    embed_size=64,
    window_size=3,
    num_negatives=5,
    batch_size=512,
    num_epochs=4,
    eval_every=100,
    checkpoint_dir=Config().checkpoint_dir / "notebook",
    hf_repo_id=None,  # never push a throwaway run
    wandb_project=None,  # no tracking either
)
print(f"random-init loss for this configuration: {random_baseline(demo, corpus.vocab_size):.3f}")

slice_ = Corpus(corpus.ids[:3_000_000], corpus.word_to_index, corpus.counts)
best = train(demo, corpus=slice_)

## 4. Evaluation

Three standard benchmarks, mirrored to the Hub so they do not depend on anyone
else's host staying up. Two details decide whether an analogy score means anything:
**all three query words are excluded** from the answer (`king - man + woman` lands
nearest `king` otherwise, so a scorer that skips this reports a near-constant zero),
and candidates are restricted to the most frequent 30,000 words.

**Coverage travels with every score.** A correlation over the 60% of pairs a small
vocabulary happens to cover is not comparable to one over 95%, and quoting the score
alone is the easiest way to overstate a result by accident.

The numbers below come from the short run above above, so they are far below the
published checkpoint's — that one scores 0.726 / 0.297 / 0.464.

In [ ]:
from ai_residency_common import load_checkpoint, resolve_required
from word_embeddings.benchmarks import evaluate_embeddings, load_benchmarks
from word_embeddings.evaluate import PUBLISHED_TEXT8, checkpoint_vocabulary

benchmarks = load_benchmarks(config)
print(
    "the short run above:",
    {k: round(best[k], 4) for k in ("wordsim353_spearman", "simlex999_spearman", "analogy_total")},
)

# The published checkpoint, for comparison -- the artifact this service serves.
path, _ = resolve_required(config.embedding_checkpoint, config.hf_repo_id)
saved = load_checkpoint(path)
scored = evaluate_embeddings(
    saved["model_state_dict"]["in_embed.weight"],
    checkpoint_vocabulary(saved["extra"]),
    benchmarks,
    max_analogy_vocab=config.max_analogy_vocab,
)
print(f"\n{'benchmark':24} {'published ckpt':>14} {'reference':>10} {'coverage':>9}")
for name, cov in (
    ("wordsim353_spearman", "wordsim353_coverage"),
    ("simlex999_spearman", "simlex999_coverage"),
    ("analogy_total", "analogy_coverage"),
):
    print(f"{name:24} {scored[name]:14.4f} {PUBLISHED_TEXT8[name]:10.2f} {scored[cov]:8.1%}")

## 5. Serving

`serving/api.py` resolves a checkpoint locally first, then from the Hub, so it works
on a host that has never trained anything. `/health` is liveness and always answers
200, so a slow or missing checkpoint never restarts a pod. `/ready` is readiness and
**retries the load** rather than reporting a fixed verdict, so a checkpoint that was
unreachable at boot is picked up when it becomes reachable.

The nearest-neighbour search goes through `word2vec.nearest_words`, which masks out
rows the vocabulary cannot name rather than letting one win the search and raise a
`KeyError` on the lookup after it.

In [ ]:
from fastapi.testclient import TestClient

from serving.api import app

# No `with`, so the lifespan never runs and nothing is loaded at import time.
client = TestClient(app)
print("/health ->", client.get("/health").json())

# /ready does not merely report that state -- it retries the load. A checkpoint
# that was unreachable at boot is picked up when it becomes reachable, instead of
# leaving the pod not-ready for the rest of its life. So this answers 200 here.
print(
    "/ready  ->", client.get("/ready").status_code, client.get("/ready").json()["checkpoint_source"]
)

In [ ]:
# With the lifespan, the published checkpoint is resolved and the endpoints answer.
with TestClient(app) as live:
    print("/ready ->", live.get("/ready").json()["checkpoint_source"])
    near = live.post("/nearest", json={"word": "river", "k": 5}).json()
    print("river ->", [w for w, _ in near["nearest"]])
    for a, b, c in (("bigger", "big", "small"), ("king", "man", "woman")):
        got = live.post("/analogy", json={"a": a, "b": b, "c": c, "k": 3}).json()
        print(f"{a} - {b} + {c} ->", [w for w, _ in got["nearest"]])

## Wrap-up

Corpus, batching, training, evaluation and serving, each calling the module the
matching `make` target runs.

The training here is a short run above, so its scores are not the service's. For the
real ones — and the 48-run sweep behind them, including the finding that CBOW takes
WordSim-353 while skip-gram takes the analogy set — see the README's Experimental
Results.

One result worth carrying away: CBOW drives its loss to 0.15 of its random-init
baseline where skip-gram reaches only 0.33, and skip-gram produces the better
vectors. The objective and the thing you want are related but not the same, which is
why the sweep ranks on benchmark score and why a loss curve alone would have picked
the wrong architecture.